# Exploratory analysis with SQL

SQLite stores the 122 scraped Falcon 9 records (2010-06-04 to 2021-06-06). These results intentionally use this table, which differs from the 90-row model sample. Eleven queries cover unique sites, initial launches, NASA CRS mass, v1.1 average, first ground landing, successful drone landings, launch totals, maximum payload, 2015 failures, rankings and site landing rates.

Historical Cape Canaveral site names are retained; six strings do not represent six physical pads.

In [1]:
"""Reproducible SpaceX capstone: scraping, SQL, visual EDA and model evaluation."""
from pathlib import Path
from io import StringIO
import calendar
import json
import re
import sqlite3
import warnings

import numpy as np
import pandas as pd
import matplotlib

import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, TimeSeriesSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, confusion_matrix


ROOT = Path.cwd().parent
DATA = ROOT / 'data'
FIG = ROOT / 'figures'


In [2]:
SQL_QUERIES = {
    "01_unique_launch_sites": "SELECT DISTINCT LaunchSite FROM launches ORDER BY LaunchSite",
    "02_ccafs_first_five": "SELECT Date, FlightNumber, LaunchSite FROM launches WHERE LaunchSite LIKE 'CCAFS%' ORDER BY Date LIMIT 5",
    "03_nasa_crs_total_payload": "SELECT COUNT(*) AS Launches, COUNT(PayloadMass) AS KnownMasses, SUM(PayloadMass) AS KnownPayloadKg FROM launches WHERE Customer LIKE '%NASA%CRS%'",
    "04_f9_v1_1_mean_payload": "SELECT COUNT(*) AS Launches, COUNT(PayloadMass) AS KnownMasses, AVG(PayloadMass) AS MeanPayloadKg FROM launches WHERE BoosterVersion LIKE 'F9 v1.1%'",
    "05_first_successful_ground_landing": "SELECT MIN(Date) AS FirstSuccessfulGroundLanding FROM launches WHERE LandingSuccess=1 AND LandingType='Ground pad'",
    "06_drone_landings_payload_4000_6000": "SELECT Date, BoosterVersion, PayloadMass FROM launches WHERE LandingSuccess=1 AND LandingType='Drone ship' AND PayloadMass>4000 AND PayloadMass<6000 ORDER BY Date",
    "07_launch_outcome_counts": "SELECT LaunchSuccess, COUNT(*) AS Launches FROM launches GROUP BY LaunchSuccess",
    "08_max_payload_boosters": "SELECT Date, BoosterVersion, PayloadMass FROM launches WHERE PayloadMass=(SELECT MAX(PayloadMass) FROM launches) ORDER BY Date",
    "09_failed_drone_landings_2015": "SELECT strftime('%m',Date) AS Month, BoosterVersion, LaunchSite, LandingOutcome FROM launches WHERE LandingResult='Failure' AND LandingType='Drone ship' AND strftime('%Y',Date)='2015' ORDER BY Date",
    "10_landing_outcomes_ranked": "SELECT LandingResult, LandingType, COUNT(*) AS Launches FROM launches WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY LandingResult, LandingType ORDER BY Launches DESC",
    "11_site_landing_success_rates": "SELECT LaunchSite, COUNT(*) AS Launches, SUM(LandingSuccess) AS SuccessfulLandings, ROUND(100.0*AVG(LandingSuccess),1) AS LandingSuccessPct FROM launches GROUP BY LaunchSite ORDER BY LandingSuccessPct DESC",
}

def sql_analysis(scraped):
    connection = sqlite3.connect(DATA / "spacex.sqlite")
    scraped.to_sql("launches", connection, if_exists="replace", index=False)
    results = {}
    for name, query in SQL_QUERIES.items():
        frame = pd.read_sql_query(query, connection)
        frame.to_csv(DATA / f"sql_{name}.csv", index=False)
        results[name] = frame.replace({np.nan: None}).to_dict(orient="records")
    (ROOT / "scripts" / "queries.sql").write_text("\n\n".join(f"-- {key}\n{value};" for key, value in SQL_QUERIES.items()), encoding="utf-8")
    connection.close()
    return results

In [3]:
scraped = pd.read_csv(DATA/'wikipedia_scraped.csv')
results = sql_analysis(scraped)
for name, query in SQL_QUERIES.items():
    print(name); print(query)
    display(pd.DataFrame(results[name]))

01_unique_launch_sites
SELECT DISTINCT LaunchSite FROM launches ORDER BY LaunchSite


,LaunchSite
0,CCAFS SLC 40
1,CCSFS SLC 40
2,Cape Canaveral LC 40
3,Cape Canaveral SLC 40
4,KSC LC 39A
5,VAFB SLC 4E


02_ccafs_first_five
SELECT Date, FlightNumber, LaunchSite FROM launches WHERE LaunchSite LIKE 'CCAFS%' ORDER BY Date LIMIT 5


,Date,FlightNumber,LaunchSite
0,2010-06-04,1.0,CCAFS SLC 40
1,2010-12-08,2.0,CCAFS SLC 40
2,2012-05-22,3.0,CCAFS SLC 40
3,2012-10-08,4.0,CCAFS SLC 40
4,2013-03-01,5.0,CCAFS SLC 40


03_nasa_crs_total_payload
SELECT COUNT(*) AS Launches, COUNT(PayloadMass) AS KnownMasses, SUM(PayloadMass) AS KnownPayloadKg FROM launches WHERE Customer LIKE '%NASA%CRS%'


...

,Month,BoosterVersion,LaunchSite,LandingOutcome
0,01,F9 v1.1 B1012,Cape Canaveral LC 40,Failure (drone ship)
1,04,F9 v1.1 B1015,Cape Canaveral LC 40,Failure (drone ship)


10_landing_outcomes_ranked
SELECT LandingResult, LandingType, COUNT(*) AS Launches FROM launches WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY LandingResult, LandingType ORDER BY Launches DESC


,LandingResult,LandingType,Launches
0,No attempt,Other / no attempt,10
1,Failure,Drone ship,5
2,Other,Ocean,5
3,Success,Drone ship,5
4,Success,Ground pad,3
5,Failure,Other / no attempt,2
6,Other,Drone ship,2


11_site_landing_success_rates
SELECT LaunchSite, COUNT(*) AS Launches, SUM(LandingSuccess) AS SuccessfulLandings, ROUND(100.0*AVG(LandingSuccess),1) AS LandingSuccessPct FROM launches GROUP BY LaunchSite ORDER BY LandingSuccessPct DESC


,LaunchSite,Launches,SuccessfulLandings,LandingSuccessPct
0,Cape Canaveral SLC 40,1,1,100.0
1,CCSFS SLC 40,12,11,91.7
2,KSC LC 39A,33,28,84.8
3,VAFB SLC 4E,16,10,62.5
4,CCAFS SLC 40,40,24,60.0
5,Cape Canaveral LC 40,20,6,30.0
